In [0]:
import sys,os
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sys.path.insert(0,os.path.abspath(os.path.join(os.getcwd(), '..')))

In [0]:
from config.config import(
    BRONZE_PATH,SOURCE_PATH,BRONZE_CUSTOMERS,BRONZE_ORDERS,BRONZE_PRODUCTS,storage_account,
)
from utils.logger import get_logger
from utils.spark_utils import add_audit_columns

In [0]:
logger.info("=" * 60)
logger.info("BRONZE LAYER INGESTION — START")
logger.info("=" * 60)

In [0]:
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope = "test-scope",key="app-id")
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope = "test-scope",key="app-secret")
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='test-scope', key='tenant-id')}/oauth2/v2.0/token"
)

dbutils.fs.ls(
    "abfss://sales-source@salessource.dfs.core.windows.net/"
)


In [0]:
logger = get_logger("01-bronze-ingestion")

### # **SOURCE-01 CUSTOMERS**

In [0]:
logger.info("Ingesting CRM customers → Bronze...")

In [0]:
schema=StructType([
  StructField("customer_id",        StringType(),  True),
        StructField("first_name",         StringType(),  True),
        StructField("last_name",          StringType(),  True),
        StructField("email",              StringType(),  True),
        StructField("phone",              StringType(),  True),
        StructField("city",               StringType(),  True),
        StructField("state",              StringType(),  True),
        StructField("country",            StringType(),  True),
        StructField("zip_code",           StringType(),  True),
        StructField("customer_segment",   StringType(),  True),
        StructField("registration_date",  StringType(),  True),
])

In [0]:
df_cus=spark.read.format('csv')\
    .option("header", "true")\
    .schema(schema)\
    .load("abfss://sales-source@salessource.dfs.core.windows.net/customers/")



In [0]:
df_cus_bronze=add_audit_columns(df_cus,BRONZE_CUSTOMERS)
df_cus_bronze.write.format("delta")\
    .mode('overwrite')\
    .save('/Volumes/salesdw/bronze/bronze_data/bronze_customers')    

In [0]:
df_bronze=spark.read.format('delta')\
    .load('/Volumes/salesdw/bronze/bronze_data/bronze_customers')



In [0]:
df_bronze.display()

## Products

In [0]:
logger.info("Ingesting Products → Bronze...")

In [0]:
df_products=spark.read.format('csv')\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://sales-source@salessource.dfs.core.windows.net/products/")



In [0]:
df_pro_bronze=add_audit_columns(df_products,'data lake folder products')
df_pro_bronze.write.format("delta")\
    .mode('overwrite')\
    .option("overwriteSchema", "true")\
    .save('/Volumes/salesdw/bronze/bronze_data/bronze_products')

In [0]:
df_bronze=spark.read.format('delta')\
    .load('/Volumes/salesdw/bronze/bronze_data/bronze_products')

In [0]:
df_bronze.display()

## orders

In [0]:
logger.info("Ingesting orders → Bronze...")

In [0]:
df_orders=spark.read.format('csv')\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://sales-source@salessource.dfs.core.windows.net/orders/")

In [0]:
df_ord_bronze=add_audit_columns(df_orders,'data lake folder orders')
df_ord_bronze.write.format("delta")\
    .mode('overwrite')\
    .option("overwriteSchema", "true")\
    .save('/Volumes/salesdw/bronze/bronze_data/bronze_orders')

In [0]:
df_bronze=spark.read.format('delta')\
    .load('/Volumes/salesdw/bronze/bronze_data/bronze_orders')

In [0]:
df_bronze.display()

In [0]:
logger.info("=" * 60)
logger.info("BRONZE LAYER INGESTION — END")
logger.info("=" * 60)

In [0]:
for tbl in [BRONZE_CUSTOMERS, BRONZE_PRODUCTS, BRONZE_ORDERS]:
        count = spark.read.format("delta").table(tbl).count()
        logger.info(f"   {tbl}: {count} rows")